# **JIRA AI CHATBOT**

Built an AI-powered JIRA chatbot using Retrieval-Augmented Generation (RAG) architecture to enable natural language querying of JIRA tickets.
Integrated semantic search with metadata filtering using ChromaDB and OpenAI embeddings to retrieve issue details such as status, priority, creation date, and resolution.
Designed structured prompt engineering to ensure accurate extraction of JIRA metadata fields and prevent hallucinations.

In [62]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Install necessary packages/dependencies**

In [ ]:
!pip install langchain-core langchain-community
!pip install -U langchain-openai
!pip install chromadb==1.5.2
!pip install langchain-text-splitters
!pip install langchain-community langchain-core
!pip install python-dotenv

**Import Libraries**

In [3]:
import pandas as pd
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import shutil
#from google.colab import files
from langchain_community.chat_models import ChatOpenAI
from dotenv import load_dotenv
import os
import re

C:\Users\shamr\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


**PHASE 1 - INGESTION**

In [4]:
file_path = "data/GFG_FINAL.csv"
df = pd.read_csv(file_path, encoding="utf-8")

In [5]:
df.head(2)

,Summary,Issue key,Issue id,Issue Type,Status,Project key,Project name,Project type,Project lead,Project description,...,Comment.75,Comment.76,Comment.77,Comment.78,Comment.79,Comment.80,Comment.81,Comment.82,Comment.83,Comment.84
0,Sourcetree repository tab width automatically ...,SRCTREEWIN-14221,1949800,Bug,Short Term Backlog,SRCTREEWIN,Sourcetree for Windows,software,rgomis,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Stashes don't show untracked files when clicki...,SRCTREEWIN-14215,1946072,Bug,Short Term Backlog,SRCTREEWIN,Sourcetree for Windows,software,rgomis,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
df.shape

(49000, 491)

In [7]:
#df.columns[0:100]
#df.columns[100:200]
#df.columns[200:300]
#df.columns[300:400]
#df.columns[400:]

In [8]:
#merge columns Comments to one column
comment_cols = [col for col in df.columns if col.startswith("Comment")]
df["All_Comments"] = df[comment_cols] \
    .apply(lambda x: " ".join(x.dropna().astype(str)), axis=1)

In [9]:
df["All_Comments"][0]

'29/May/2023 10:57 AM;698877135425;Hi [~b80d91f3d253]\xa0Attached image is not visible.Please upload it to google drive and share the link.Please share your screen resolution and scale information as shown in the following image:!image-2023-05-29-16-26-21-583.png|width=637,height=310!;;; 29/May/2023 12:00 PM;b80d91f3d253;I try, for now, to attach new images in this comment.My screen dataThe size of text: 100%Display resolution: 1920 x 1080.Note that the problem is only with short uppercase project names, like as "JMS". I think that the Sourcetree code calculates the tab with as a fixed size of single char mulipling it for the total length of the name. For outer my tabs with name with all lowercase letters, the problem don\'t happens.Name: "jps_application" -> good: there is some space between the name and the closing "X".Name "JMW" -> bad: there is not enough space between the name and the closing "X"; when mouse over, the "X" overwrite the last letter of the name.\xa0\xa0!image-2023-0

In [10]:
#final Dataframe
df_final = df[['Summary','Issue key','Issue id','Issue Type', 'Status',
       'Project key', 'Project name', 'Project type', 'Project url', 'Priority',
       'Resolution', 'Created', 'Updated', 'Last Viewed',
       'Resolved', 'Description','Custom field (Symptom Severity)','All_Comments']]

In [11]:
df_final.head(2)

,Summary,Issue key,Issue id,Issue Type,Status,Project key,Project name,Project type,Project url,Priority,Resolution,Created,Updated,Last Viewed,Resolved,Description,Custom field (Symptom Severity),All_Comments
0,Sourcetree repository tab width automatically ...,SRCTREEWIN-14221,1949800,Bug,Short Term Backlog,SRCTREEWIN,Sourcetree for Windows,software,https://www.sourcetreeapp.com,Low,NaN,29/May/2023 6:43 AM,29/May/2023 2:06 PM,31/May/2023 7:54 AM,NaN,SourceTree 3.4.13 [31 May 2023Change: Sourcetr...,Severity 3 - Minor,29/May/2023 10:57 AM;698877135425;Hi [~b80d91f...
1,Stashes don't show untracked files when clicki...,SRCTREEWIN-14215,1946072,Bug,Short Term Backlog,SRCTREEWIN,Sourcetree for Windows,software,https://www.sourcetreeapp.com,Low,NaN,25/May/2023 7:36 PM,30/May/2023 4:21 AM,31/May/2023 7:54 AM,NaN,I could swear that in the past if I created a ...,Severity 3 - Minor,25/May/2023 7:39 PM;9fe2303fdef9;This also mak...


In [12]:
df_final.shape

(49000, 18)

**Data Preprocessing - Removing Null values**

In [13]:
df_final.isna().sum()

Summary                                0
Issue key                              0
Issue id                               0
Issue Type                             0
Status                                 0
Project key                            0
Project name                           0
Project type                           0
Project url                            0
Priority                            6713
Resolution                         32438
Created                                0
Updated                                0
Last Viewed                            0
Resolved                           32438
Description                            0
Custom field (Symptom Severity)     6762
All_Comments                           0
dtype: int64

In [14]:
#fill na
df_final['Priority'] = df_final['Priority'].fillna("")
df_final['Resolution'] = df_final['Resolution'].fillna("")
df_final['Resolved'] = df_final['Resolved'].fillna("")
df_final['Custom field (Symptom Severity)'] = df_final['Custom field (Symptom Severity)'].fillna("")

C:\Users\shamr\AppData\Local\Temp\ipykernel_31064\15098841.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['Priority'] = df_final['Priority'].fillna("")
C:\Users\shamr\AppData\Local\Temp\ipykernel_31064\15098841.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['Resolution'] = df_final['Resolution'].fillna("")
C:\Users\shamr\AppData\Local\Temp\ipykernel_31064\15098841.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_

In [15]:
df_final.isna().sum().sum()

0

**Create document for Retrieval**

In [16]:
docs = []

for _ ,row in df_final.iterrows():
    # Create searchable content
    content = f"""
    Summary: {row['Summary']}
    Description: {row['Description']}
    All_Comments:{row["All_Comments"]}
    """

    # Create metadata dictionary
    metadata = {
        "issue_key": row["Issue key"],
        "issue_id": row["Issue id"],
        "issue_type": row["Issue Type"],
        "status": row["Status"],
        "priority": row["Priority"],
        "project": row["Project name"],
        "Project key":row["Project key"],
        "Project type": row["Project type"],
        "Project url":row["Project url"],
        "Resolution":row["Resolution"],
        "Created":row["Created"],
        "Updated":row["Updated"],
        "Last Viewed":row["Last Viewed"],
        "Resolved":row["Resolved"],
        "Custom field (Symptom Severity)":row["Custom field (Symptom Severity)"],

    }
    docs.append(
        Document(
            page_content=content,
            metadata=metadata
        )
    )

**Chunking**

In [17]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=550,
    chunk_overlap=100
)

chunked_docs = text_splitter.split_documents(docs)

In [18]:
len(chunked_docs)

190414

**Embedding**

In [5]:
embedding = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    encode_kwargs={"normalize_embeddings": True}
)

**Storage to VectorDB - ChromaDB**

In [ ]:
persist_directory = "./chroma_db"

# Initialize empty vector store
vectorstore = Chroma(
    persist_directory=persist_directory,
    embedding_function=embedding
)

batch_size = 200  # You can tune this (50–200 is safe)

for i in range(0, len(chunked_docs), batch_size):
    batch = chunked_docs[i:i + batch_size]

    print(f"Processing batch {i//batch_size + 1} "
          f"of {len(chunked_docs)//batch_size + 1}")

    vectorstore.add_documents(batch)

# Save to disk
vectorstore.persist()

print("✅ All documents stored successfully.")

/tmp/ipython-input-1041/676610392.py:6: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


Processing batch 1 of 953
Processing batch 2 of 953
Processing batch 3 of 953
Processing batch 4 of 953
Processing batch 5 of 953
Processing batch 6 of 953
Processing batch 7 of 953
Processing batch 8 of 953
Processing batch 9 of 953
Processing batch 10 of 953
Processing batch 11 of 953
Processing batch 12 of 953
Processing batch 13 of 953
Processing batch 14 of 953
Processing batch 15 of 953
Processing batch 16 of 953
Processing batch 17 of 953
Processing batch 18 of 953
Processing batch 19 of 953
Processing batch 20 of 953
Processing batch 21 of 953
Processing batch 22 of 953
Processing batch 23 of 953
Processing batch 24 of 953
Processing batch 25 of 953
Processing batch 26 of 953
Processing batch 27 of 953
Processing batch 28 of 953
Processing batch 29 of 953
Processing batch 30 of 953
Processing batch 31 of 953
Processing batch 32 of 953
Processing batch 33 of 953
Processing batch 34 of 953
Processing batch 35 of 953
Processing batch 36 of 953
Processing batch 37 of 953
Processing

/tmp/ipython-input-1041/676610392.py:22: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


In [ ]:
#To download vector db to local
shutil.make_archive("chroma_db_backup", 'zip', "chroma_db")
files.download("chroma_db_backup.zip")

'/content/chroma_db_backup.zip'

In [6]:
# Load existing Chroma DB
vectorstore = Chroma(
    persist_directory=r"C:\Users\shamr\OneDrive\Shamreen\ML\ML JH\Projects (R)\Jira AI Chatbot\chroma_db_backup",  # your stored folder
    embedding_function=embedding
)

**PHASE 2,3 - RETRIEVAL AND AUGMENTATION**

**Retrieval**

In [ ]:
query= "What is the current status of issue SRCTREEWIN-14037"
# Retrieve similar docs
retrieve_docs = vectorstore.similarity_search(query, k=10)
print(retrieve_docs)

context = "\n\n".join([doc.page_content for doc in retrieve_docs])

In [ ]:
print("Total docs in DB:", vectorstore._collection.count())

In [31]:
query= "Give me details of issue with Displaying all changes between hash1 and hash2 doesn't work"
# Retrieve similar docs
retrieve_docs = vectorstore.similarity_search(query, k=10)
print(retrieve_docs)

context = "\n\n".join([doc.page_content for doc in retrieve_docs])

[]


**LLM Generation [Augmentation]**

In [23]:
env_path = "/content/drive/MyDrive/1. Introduction - RAG/key.env"
load_dotenv(dotenv_path=env_path)
apiKey = os.getenv("OPENAI_API_KEY")

In [24]:
llm = ChatOpenAI(model="gpt-4",api_key=apiKey)

C:\Users\shamr\anaconda3\lib\site-packages\langchain_core\_api\deprecation.py:119: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 0.2.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import ChatOpenAI`.
  warn_deprecated(


In [ ]:
def retrieve_docs_no(query):

    # ==============================
    # 🔥 STEP 0: DATE QUERY HANDLING
    # ==============================
    date_match = re.search(r"\d{1,2}/[A-Za-z]{3}/\d{4}", query)

    if date_match:
        target_date = date_match.group()

        docs = vectorstore.similarity_search("", k=200)

        filtered_docs = []

        for doc in docs:
            comments = doc.page_content or ""

            if target_date.lower() in comments.lower():
                filtered_docs.append(doc)

        if len(filtered_docs) == 0:
            return f"No issues found with comments on {target_date}"

        # Return unique issue keys
        issue_keys = list(set([
            doc.metadata.get("issue_key") for doc in filtered_docs
        ]))

        return "\n".join(issue_keys)


    # ==============================
    # 🔹 STEP 1: TICKET QUERY
    # ==============================
    match = re.search(r"[A-Z]+-\d+", query)

    if match:
        issue_key = match.group()

        docs = vectorstore.similarity_search(
            query="",
            filter={"issue_key": issue_key},
            k=5
        )

        if len(docs) == 0:
            return f"Issue {issue_key} not found in dataset."


    # ==============================
    # 🔹 STEP 2: SEMANTIC QUERY (RAG)
    # ==============================
    else:
        queries = generate_queries(query)[:3]

        all_docs = []

        for q in queries:
            results = hybrid_search_with_scores(q, k=40)

            filtered_docs = dynamic_top_k(results, query, max_k=10)

            all_docs.extend(filtered_docs)

        # 🔹 Fallback
        if len(all_docs) == 0:
            print("⚠️ Fallback triggered")

            fallback = hybrid_search_with_scores(query, k=5)
            all_docs = [doc for doc, _ in fallback]

        # 🔹 Deduplicate
        clean_docs = [
            doc if not isinstance(doc, tuple) else doc[0]
            for doc in all_docs
        ]

        unique_docs = list({
            doc.page_content: doc for doc in clean_docs
        }.values())

        # 🔹 Re-rank
        docs = rerank_documents(query, unique_docs, top_k=5)


    # ==============================
    # 🔹 STEP 3: BUILD CONTEXT
    # ==============================
    context = ""

    for doc in docs:
        context += f"""
Issue Key: {doc.metadata.get('issue_key')}
Issue Type: {doc.metadata.get('issue_type')}
Status: {doc.metadata.get('status')}
Project name: {doc.metadata.get('project')}
Project type: {doc.metadata.get('Project type')}
Project url: {doc.metadata.get('Project url')}
Priority: {doc.metadata.get('priority')}
Resolution: {doc.metadata.get('Resolution')}
Created: {doc.metadata.get('Created')}
Updated: {doc.metadata.get('Updated')}
Last Viewed: {doc.metadata.get('Last Viewed')}
Resolved: {doc.metadata.get('Resolved')}
Custom field (Symptom Severity): {doc.metadata.get('Custom field (Symptom Severity)')}

Description: {doc.page_content}
"""


    # ==============================
    # 🔹 STEP 4: INTENT DETECTION
    # ==============================
    if any(word in query.lower() for word in ["why", "reason", "analysis"]):
        mode = "analysis"
    else:
        mode = "extraction"


    # ==============================
    # 🔹 STEP 5: PROMPT
    # ==============================
    if mode == "analysis":
        prompt = f"""
You are a JIRA analyst.

Analyze the context and identify possible reasons.

Look for:
- unresolved tickets
- pending status
- missing resolution
- repeated updates
- high priority not closed

If exact reason is not explicitly stated, infer from patterns.

CONTEXT:
{context}

QUESTION:
{query}

Give a concise answer.
"""
    else:
        prompt = f"""
You are a JIRA structured data assistant.

Extract ONLY requested fields.

RULES:
- Use only provided context
- Do not infer
- Do not add extra text

CONTEXT:
{context}

QUESTION:
{query}

OUTPUT FORMAT:
<Field Name>: <Exact Value>

If not found:
No issue found.
"""

    response = llm.invoke(prompt)
    return response.content

In [1]:
def retrieve_docs(query):
    match = re.search(r"[A-Z]+-\d+", query)
    if match:
        issue_key = match.group()
        retrieve_docs_result = vectorstore.similarity_search(
            query="",
            filter={"issue_key": issue_key},
            k=5
        )
    else:
        retrieve_docs_result = vectorstore.similarity_search(
            query,
            k=5
        )

    context = ""

    for doc in retrieve_docs_result:
        context += f"""
Issue Key: {doc.metadata.get('issue_key')}
Issue Type: {doc.metadata.get('issue_type')}
Status: {doc.metadata.get('status')}
Project name: {doc.metadata.get('project')}
Project type: {doc.metadata.get('Project type')}
Project url: {doc.metadata.get('Project url')}
Priority: {doc.metadata.get('priority')}
Resolution: {doc.metadata.get('Resolution')}
Created: {doc.metadata.get('Created')}
Updated: {doc.metadata.get('Updated')}
Last Viewed: {doc.metadata.get('Last Viewed')}
Resolved: {doc.metadata.get('Resolved')}
Custom field (Symptom Severity): {doc.metadata.get('Custom field (Symptom Severity)')}

Description: {doc.page_content}
"""

    if any(word in query.lower() for word in ["why", "reason", "analysis"]):
        mode = "analysis"
    else:
        mode = "extraction"

    if mode == "analysis":
        prompt = f"""
You are a JIRA analyst.

Analyze the context and identify possible reasons.

Look for:
- unresolved tickets
- pending status
- missing resolution
- repeated updates
- high priority not closed

If exact reason is not explicitly stated, infer from patterns.

CONTEXT:
{context}

QUESTION:
{query}

Give a concise answer.
"""
    else:
        prompt = f"""
You are a JIRA structured data assistant.

Your task is to extract requested fields strictly from the provided context.

RULES:
1. Use ONLY the information present in the context.
2. Do NOT infer, assume, or generate missing values.
3. If a requested field exists in the context, you MUST return it.
4. If a requested field does not exist, return: Not Available
5. Do NOT summarize.
6. Do NOT explain.
7. Do NOT add extra text.
8. If output has repeated issue number, do not repeat it.

FIELD EXTRACTION RULE:
Return ONLY the fields that are explicitly requested in the user question.
Preserve exact field names and exact values as written in context.
Do NOT change date formats.

CONTEXT:
{context}

USER QUESTION:
{query}

OUTPUT FORMAT:
<Field Name>: <Exact Value>

If no matching issue is found in the context, return:
No issue found.
"""
    response = llm.invoke(prompt)
    return response.content

IndentationError: expected an indented block after 'for' statement on line 19 (3040370256.py, line 20)

In [25]:
def retrieve_docs_old(query):
  match = re.search(r"[A-Z]+-\d+", query)
  if match:
    issue_key = match.group()
    retrieve_docs = vectorstore.similarity_search(
        query="",
        filter={"issue_key": issue_key},
        k=5
        )
  else:
    retrieve_docs = vectorstore.similarity_search(
    query,
    k=5
  )

  context = ""

  for doc in retrieve_docs:
    context += f"""
  Issue Key: {doc.metadata.get('issue_key')}
  Issue Type: {doc.metadata.get('issue_type')}
  Status: {doc.metadata.get('status')}
  Project name: {doc.metadata.get('project')}
  Project type: {doc.metadata.get('Project type')}
  Project url: {doc.metadata.get('Project url')}
  Priority: {doc.metadata.get('priority')}
  Resolution: {doc.metadata.get('Resolution')}
  Created: {doc.metadata.get('Created')}
  Updated: {doc.metadata.get('Updated')}
  Last Viewed: {doc.metadata.get('Last Viewed')}
  Resolved: {doc.metadata.get('Resolved')}
  Custom field (Symptom Severity): {doc.metadata.get('Custom field (Symptom Severity)')}

  Description: {doc.page_content}
  """

  prompt = f"""
You are a JIRA structured data assistant.

Your task is to extract requested fields strictly from the provided context.

RULES:
1. Use ONLY the information present in the context.
2. Do NOT infer, assume, or generate missing values.
3. If a requested field exists in the context, you MUST return it.
4. If a requested field does not exist, return: Not Available
5. Do NOT summarize.
6. Do NOT explain.
7. Do NOT add extra text.
8. If output has repeated issue number , Do not repeat it .

FIELD EXTRACTION RULE:
Return ONLY the fields that are explicitly requested in the user question.
Preserve exact field names and exact values as written in context.
Do NOT change date formats.

CONTEXT:
{context}

USER QUESTION:
{query}

OUTPUT FORMAT:
<Field Name>: <Exact Value>

If no matching issue is found in the context, return:
No issue found.
"""
  response = llm.invoke(prompt)
  return response.content

In [26]:
query= '''Is this issue Displaying all changes between hash1 and
hash2 doesn't raised?'''
print(retrieve_docs(query))

No issue found.


In [28]:
query= '''What is the current status of issue SRCTREEWIN-14037'''
print(retrieve_docs(query))

No issue found.


In [141]:
query= '''Give me the list of issues which were commented on 25/May/2023 ?'''
print(retrieve_docs(query))

Issue Key: SRCTREEWIN-13997
Description: All_Comments:25/May/2023 6:00 AM;698877135425;We are looking into this issue.;;;


In [142]:
query= '''Any issue asking to Please upload the image to Google Drive and share the link?'''
print(retrieve_docs(query))

Issue Key: SRCTREEWIN-13731
Issue Type: Bug
Status: Needs Triage
Project name: Sourcetree for Windows
Project type: software
Project url: https://www.sourcetreeapp.com
Priority: Low
Resolution: Not Available
Created: 22/Nov/2021 11:43 AM
Updated: 08/Dec/2021 4:21 AM
Last Viewed: 31/May/2023 8:34 AM
Resolved: Not Available
Custom field (Symptom Severity): Minor
Description: upload on google drive and share the link. ;;; 02/Dec/2021 1:52 PM;7b474805d6c5;Here it is https://drive.google.com/file/d/1k0TKNMKRXGDRZ9cOBbc9FSqHhUX6UCwt/view?usp=sharing;;; 03/Dec/2021 7:39 AM;e85ff1f4514c;Hi [~7b474805d6c5] , It is not accessible to me. Please allow. Also please provide info like Sourcetree version, SCM provider like github or bitbucket, and log file to analysis in details.  ;;; 07/Dec/2021 2:17 PM;7b474805d6c5;Hello VipinI think i already gave permission on the snapshotthe version of sourcetree is 3.4.7.0SCM provider is


In [143]:
query= ''' Any issue raised App freezes and displays "Not Responding" in titlebar constantly?
If yes , what is the creation date'''
print(retrieve_docs(query))

Issue Key: SRCTREEWIN-13984
Created: 30/Jan/2023 6:17 PM
